# 04 - U-Net Segmentation

This notebook trains a U-Net-style model to predict Oxford-IIIT Pet trimap masks.

## Google Colab Setup

Run the next cell only when using Google Colab. It mounts Google Drive, moves into the project folder, and installs the extra packages Colab may not already include. If you run locally, skip it.


In [ ]:
# Colab-only setup. Skip this cell when running locally.
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/PetVision-DeepLearning')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'tensorflow-datasets', 'seaborn', 'scikit-learn', 'streamlit', 'opencv-python'
    ])
    print('Colab project root:', os.getcwd())
else:
    print('Not running in Google Colab. Continue with the local setup cells below.')


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

for path in [
    "models",
    "results/classification",
    "results/segmentation",
    "results/gradcam",
    "results/figures",
]:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_loader import get_splits, configure_for_performance
from src.preprocessing import preprocess_segmentation, apply_segmentation_random_flip
from src.models_segmentation import build_unet, compile_segmenter
from src.training import standard_callbacks
from src.evaluation import evaluate_segmentation_model
from src.visualization import plot_training_history, show_segmentation_triplet

In [ ]:
BATCH_SIZE = 16
EPOCHS = 15
IMAGE_SIZE = (160, 160)

train_raw, val_raw, test_raw, info, label_names = get_splits()

def prep(example):
    return preprocess_segmentation(example, image_size=IMAGE_SIZE)

train_ds = train_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.map(apply_segmentation_random_flip, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = configure_for_performance(train_ds, BATCH_SIZE, shuffle=True)
val_ds = configure_for_performance(val_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE)
test_ds = configure_for_performance(test_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE)

In [ ]:
model = build_unet(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), num_classes=3)
compile_segmenter(model, learning_rate=1e-3, from_logits=True)
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=standard_callbacks(PROJECT_ROOT / "models/best_segmenter.keras", patience=4),
)
plot_training_history(history, PROJECT_ROOT / "results/segmentation/segmentation_training_curves.png", "U-Net Segmentation")

In [ ]:
best_segmenter = tf.keras.models.load_model(PROJECT_ROOT / "models/best_segmenter.keras")
metrics = evaluate_segmentation_model(best_segmenter, test_ds)
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(PROJECT_ROOT / "results/segmentation/segmentation_metrics.csv", index=False)
metrics_df

In [ ]:
# Visualize predictions.
images, masks = next(iter(test_ds))
logits = best_segmenter.predict(images, verbose=0)
pred_masks = np.argmax(logits, axis=-1)

for i in range(4):
    show_segmentation_triplet(
        images[i].numpy(),
        masks[i].numpy(),
        pred_masks[i],
        PROJECT_ROOT / f"results/segmentation/mask_example_{i}.png",
    )

show_segmentation_triplet(
    images[0].numpy(),
    masks[0].numpy(),
    pred_masks[0],
    PROJECT_ROOT / "results/segmentation/mask_examples.png",
)